<a href="https://colab.research.google.com/github/eniyigena/ML-Teaching-/blob/main/energy_transition_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Global Electricity Transition — Exploratory Data Analysis

**Research question:** Is renewable electricity growth replacing fossil-fuel generation, or can both grow together?

**Source:** Our World in Data, *OWID Energy Dataset* (release 2026-09-10): https://catalog.ourworldindata.org/garden/energy/2026-09-10/owid_energy/owid_energy.csv

**Observation unit:** Location–year. The dataset includes individual countries and geographic aggregates; these must not be summed together.

**Scope:** Electricity generation (TWh and shares), not total energy use. This notebook documents the analysis behind the five panels produced by `final_visualization.py`.

In [ ]:
import sys
print(sys.executable)

import pandas
import altair
import vl_convert
print("Required packages are available.")

c:\Users\robot\AppData\Local\Altair\MF\python.exe
Required packages are available.


In [ ]:
# Load the exact source used by the final visualization script.
from pathlib import Path
import pandas as pd
import numpy as np
import altair as alt

SOURCE_URL = 'https://catalog.ourworldindata.org/garden/energy/2026-09-10/owid_energy/owid_energy.csv'
LOCAL_CSV = Path('owid_energy.csv')
if LOCAL_CSV.exists():
    df = pd.read_csv(LOCAL_CSV)
else:
    df = pd.read_csv(SOURCE_URL)
    df.to_csv(LOCAL_CSV, index=False)

print('Dataset shape:', df.shape)
display(df.head())

Dataset shape: (19494, 130)


,country,year,iso_code,population,gdp,bioenergy_electricity_per_capita_kwh,bioenergy_electricity_share_pct,bioenergy_electricity_twh,biofuels_energy_annual_change_pct,biofuels_energy_annual_change_twh,...,total_energy_per_gdp_kwh_per_dollar,total_energy_twh,wind_electricity_per_capita_kwh,wind_electricity_share_pct,wind_electricity_twh,wind_energy_annual_change_pct,wind_energy_annual_change_twh,wind_energy_per_capita_kwh,wind_energy_share_pct,wind_energy_twh
0,ASEAN (Ember),2000,NaN,NaN,NaN,NaN,1.549794,5.87,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN
1,ASEAN (Ember),2001,NaN,NaN,NaN,NaN,1.594707,6.46,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN
2,ASEAN (Ember),2002,NaN,NaN,NaN,NaN,1.528198,6.62,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN
3,ASEAN (Ember),2003,NaN,NaN,NaN,NaN,1.623391,7.44,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN
4,ASEAN (Ember),2004,NaN,NaN,NaN,NaN,1.689455,8.39,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN


## 1. Inspect the dataset and its structure

**Question:** Which years and locations are represented, and are there duplicate location–year observations? We must identify country records separately from the `World` aggregate.

In [ ]:
print('Year range:', df.year.min(), '-', df.year.max())
print('Distinct locations:', df.country.nunique())

# Check the number of ISO-coded countries and duplicated location-year pairs
country_mask = df["iso_code"].fillna("").str.fullmatch(r"[A-Z]{3}")
countries = df.loc[country_mask].copy()

print(f"Three-letter ISO-coded locations: {countries['iso_code'].nunique()}")
print(
    "Duplicate location-year rows:",
    df.duplicated(subset=["country", "year"]).sum())

Year range: 1700 - 2025
Distinct locations: 260
Three-letter ISO-coded locations: 220
Duplicate location-year rows: 0


### Variables relevant to the research question

Before narrowing the data, examine the distributions of electricity generation, source shares, and renewable technologies.

In [ ]:
key = [
    "electricity_generation_twh",
    "renewables_electricity_share_pct",
    "fossil_fuels_electricity_share_pct",
    "renewables_electricity_twh",
    "fossil_fuels_electricity_twh",
    "solar_electricity_twh",
    "wind_electricity_twh",
    "hydro_electricity_twh",
]
display(df[key].describe().T.round(2))

,count,mean,std,min,25%,50%,75%,max
electricity_generation_twh,7871.0,789.55,2633.35,0.0,2.19,21.82,169.67,31772.35
renewables_electricity_share_pct,7615.0,29.31,30.21,0.0,3.46,18.36,47.62,100.00
fossil_fuels_electricity_share_pct,7315.0,65.05,31.03,0.0,43.16,70.91,94.28,100.00
renewables_electricity_twh,8463.0,177.61,612.10,0.0,0.21,4.48,42.26,10726.49
fossil_fuels_electricity_twh,7341.0,525.80,1741.73,0.0,0.65,10.43,129.29,18275.51
solar_electricity_twh,8536.0,12.25,93.74,0.0,0.00,0.00,0.05,2778.64
wind_electricity_twh,8383.0,23.73,134.61,0.0,0.00,0.00,0.24,2713.14
hydro_electricity_twh,9359.0,118.04,368.20,0.0,0.09,3.36,31.34,4434.97


**Why inspect missingness?** Historical coverage is not uniform. Missing values must not be interpreted as zero, and the selection of comparison years should be justified by the available observations.

In [ ]:
# Check for missing values in the key features
print('Missing (%), entire dataset:')
modern = countries.loc[countries["year"].between(2000, 2024)].copy()
missing = pd.DataFrame({
    "All historical rows (%)": df[key].isna().mean() * 100,
    "Countries, 2000-2024 (%)": modern[key].isna().mean() * 100,
}).round(1)
display(missing.sort_values("Countries, 2000-2024 (%)", ascending=False))

Missing (%), entire dataset:


,All historical rows (%),"Countries, 2000-2024 (%)"
wind_electricity_twh,57.0,6.9
hydro_electricity_twh,52.0,4.8
renewables_electricity_share_pct,60.9,3.6
fossil_fuels_electricity_share_pct,62.5,3.6
solar_electricity_twh,56.2,3.5
electricity_generation_twh,59.6,3.1
renewables_electricity_twh,56.6,3.1
fossil_fuels_electricity_twh,62.3,3.1


## 2. What years are comparable across countries?
**Question:** Does the newest year provide the broadest country coverage?  
**Reason:** The newest year can have partial reporting. Country-level time comparisons should use a common pair of years and exclude unreported observations without treating missing as zero.  
**Decision:** Reserve 2025 for global context; use **2010–2024** for cross-country comparisons after checking country counts.

In [ ]:
coverage = countries.groupby("year").agg(
    locations=("iso_code", "nunique"),
    with_total=("electricity_generation_twh", "count"),
    with_renewables=("renewables_electricity_twh", "count"),
    with_fossils=("fossil_fuels_electricity_twh", "count"),
)
display(coverage.loc[[2000, 2010, 2020, 2023, 2024, 2025]])
print('Why 2024 instead of 2025? Compare their country coverage above.')

,locations,with_total,with_renewables,with_fossils
year,,,,
2000,216,211,211,211
2010,218,212,212,212
2020,220,213,213,213
2023,220,212,212,212
2024,220,196,196,196
2025,126,90,90,90


Why 2024 instead of 2025? Compare their country coverage above.


## 3. Is the global electricity mix changing?
**Question:** Is the share of renewables increasing, and the fossil share decreasing?  
**Method:** Use the supplied `World` aggregate, not a sum of countries (which may have gaps and aggregation overlaps). Plot shares as **percentages**, not TWh.

In [ ]:
world = df.loc[
    (df["country"] == "World") & df["year"].between(2000, 2025)
].sort_values("year").copy()

share_cols = {
    "renewables_electricity_share_pct": "Renewables",
    "fossil_fuels_electricity_share_pct": "Fossil fuels",
}
trend = world[["year", *share_cols]].melt(
    id_vars="year", var_name="source", value_name="share_pct"
)
trend["source"] = trend["source"].map(share_cols)
trend = trend.dropna(subset=["share_pct"])

chart1 = alt.Chart(trend).mark_line(strokeWidth=3).encode(
    x=alt.X("year:Q", title="Year", axis=alt.Axis(format="d")),
    y=alt.Y("share_pct:Q", title="Share of global electricity (%)", scale=alt.Scale(domain=[0, 80])),
    color=alt.Color("source:N", scale=alt.Scale(
        domain=["Renewables", "Fossil fuels"], range=["#18896b", "#cb713f"]
    ), title=None),
    tooltip=["year:Q", "source:N", alt.Tooltip("share_pct:Q", format=".1f")],
).properties(width=650, height=260, title="Global electricity mix, 2000–2025")
display(chart1)

display(world.loc[world["year"].isin([2000, 2025]),
    ["year", *share_cols, "renewables_electricity_twh", "fossil_fuels_electricity_twh"]])

alt.Chart(...)

,year,renewables_electricity_share_pct,fossil_fuels_electricity_share_pct,renewables_electricity_twh,fossil_fuels_electricity_twh
19078,2000,18.716286,64.656342,2859.620117,9878.700195
19103,2025,33.760456,57.390434,10726.490234,18234.289062


**Interpretation:** Compare the renewable and fossil *shares* above with their total generation in TWh. Falling fossil share need not mean falling absolute fossil generation when total demand grows.

**Follow-up:** Which technologies account for the increase in renewable generation? Cite actual values from the output when presenting the results; avoid treating the chart as causal evidence.

## 4. Which renewable technologies drive growth?
**Reason:** A single renewable total hides differences between hydropower, wind, solar, bioenergy and other renewable technologies. These five components are non-overlapping; **do not** add `other_renewables_including_bioenergy` to bioenergy because that would double count.

In [ ]:
technology_cols = {
    "hydro_electricity_twh": "Hydro",
    "wind_electricity_twh": "Wind",
    "solar_electricity_twh": "Solar",
    "bioenergy_electricity_twh": "Bioenergy",
    "other_renewables_excluding_bioenergy_electricity_twh": "Other renewables",
}

# Check whether the five components are recorded for every year we plot.
missing_technology = world[["year", *technology_cols]].loc[
    world[list(technology_cols)].isna().any(axis=1)
]
print("Years with incomplete technology data:", missing_technology["year"].tolist())
if not missing_technology.empty:
    raise ValueError("Review missing global technologies before making a stacked chart.")

tech = world[["year", *technology_cols]].melt(
    id_vars="year", var_name="source", value_name="twh"
).dropna(subset=["twh"])
tech["source"] = tech["source"].map(technology_cols)
order = ["Hydro", "Wind", "Solar", "Bioenergy", "Other renewables"]
tech["stack_order"] = tech["source"].map({name: i for i, name in enumerate(order)})

chart2 = alt.Chart(tech).mark_area().encode(
    x=alt.X("year:Q", title="Year", axis=alt.Axis(format="d")),
    y=alt.Y("sum(twh):Q", title="Renewable generation (TWh)", stack="zero"),
    color=alt.Color("source:N", title="Technology", scale=alt.Scale(
        domain=["Hydro", "Wind", "Solar", "Bioenergy", "Other renewables"],
        range=["#4f87af", "#18896b", "#efb445", "#9d83b0", "#9ab4bf"])),
    order=alt.Order("stack_order:Q", sort="ascending"),
    tooltip=["year:Q", "source:N", alt.Tooltip("sum(twh):Q", format=",.1f")]
).properties(width=650, height=260, title="Global renewable generation by technology")
display(chart2)

Years with incomplete technology data: []


alt.Chart(...)

**Follow-up:** Global growth does not tell us whether every country is moving similarly. Which major electricity producers increased their renewable share most?

## 5. How do major electricity producers compare?
**Method:** Join each country's 2010 and 2024 records using ISO code; require complete observations in both years; use the 15 largest **2024 electricity producers** among eligible countries. A change in percentage points equals the new share minus the old share; it is *not* percentage growth.

In [ ]:
measures = [
    "electricity_generation_twh",
    "renewables_electricity_share_pct",
    "renewables_electricity_twh",
    "fossil_fuels_electricity_twh",
]
base = countries.loc[countries["year"] == 2010, ["iso_code", "country", *measures]]
end = countries.loc[countries["year"] == 2024, ["iso_code", *measures]]
matched = base.merge(end, on="iso_code", suffixes=("_2010", "_2024"), validate="one_to_one")

# Only require values needed for the planned comparisons.
required = [f"{col}_{year}" for col in measures for year in (2010, 2024)]
matched = matched.dropna(subset=required).copy()
print('Paired 2010/2024 countries with complete required values:', len(matched))

matched["share_change_pp"] = (
    matched["renewables_electricity_share_pct_2024"]
    - matched["renewables_electricity_share_pct_2010"]
)
matched["renewable_change_twh"] = (
    matched["renewables_electricity_twh_2024"]
    - matched["renewables_electricity_twh_2010"]
)
matched["fossil_change_twh"] = (
    matched["fossil_fuels_electricity_twh_2024"]
    - matched["fossil_fuels_electricity_twh_2010"]
)

# Retain the largest producers by their 2024 generation.
top15 = matched.nlargest(15, "electricity_generation_twh_2024").copy()
top30 = matched.nlargest(30, "electricity_generation_twh_2024").copy()
print("Matched locations:", len(matched))
print("Top-15 comparison sample:", len(top15))
print("Top-30 comparison sample:", len(top30))
display(top15[["country", "share_change_pp", "renewable_change_twh", "fossil_change_twh"]].round(1))

Paired 2010/2024 countries with complete required values: 194
Matched locations: 194
Top-15 comparison sample: 15
Top-30 comparison sample: 30


,country,share_change_pp,renewable_change_twh,fossil_change_twh
40,China,15.0,2612.5,2891.2
207,United States,13.7,632.1,-330.0
92,India,4.7,261.9,806.7
163,Russia,2.0,51.8,74.1
100,Japan,12.7,114.9,-47.1
26,Brazil,2.5,213.9,14.8
34,Canada,3.2,38.6,-4.3
183,South Korea,8.6,55.0,35.2
69,France,13.5,76.0,-28.5
75,Germany,41.8,185.4,-173.1


In [ ]:
chart3 = alt.Chart(top15).mark_bar().encode(
    y=alt.Y("country:N", sort="-x", title=None),
    x=alt.X("share_change_pp:Q", title="Change in renewable share (percentage points)", scale=alt.Scale(zero=True)),
    color=alt.condition("datum.share_change_pp >= 0",
                        alt.value("#18896b"), alt.value("#cb713f")),
    tooltip=["country:N", alt.Tooltip("share_change_pp:Q", format="+.1f")]
).properties(width=620, height=360,
    title="Renewable-share change among 15 major electricity producers, 2010–2024")
zero_bar = alt.Chart(pd.DataFrame({"zero": [0]})).mark_rule(
    color="#879aa7"
).encode(x="zero:Q")
display(chart3 + zero_bar)

alt.LayerChart(...)

**Interpretation:** Compare the largest eligible electricity producers, ranked by their *2024 total generation*. Measure share change in **percentage points (pp)**, not percent. A country may generate more renewable electricity while its renewable *share* falls if overall electricity production rises faster.

**Follow-up:** To test whether renewable growth coincided with lower fossil generation, compare changes in both absolute quantities.

## 6. Do added renewables coincide with less fossil power?
**Why a scatter plot?** Each point is one country and each axis represents a different numerical change. Zero lines divide the results into intuitive categories. Positive renewable change and negative fossil change is the clearest descriptive pattern consistent with substitution, **not proof of causality**.

In [ ]:
top30["pattern"] = np.select(
    [
        (top30["renewable_change_twh"] > 0) & (top30["fossil_change_twh"] < 0),
        (top30["renewable_change_twh"] > 0) & (top30["fossil_change_twh"] >= 0),
    ],
    ["Renewables up, fossils down", "Both increased"],
    default="Other pattern"
)
print(top30["pattern"].value_counts())

scatter = alt.Chart(top30).mark_circle(size=95, opacity=0.85).encode(
    x=alt.X("renewable_change_twh:Q", title="Change in renewables (TWh)"),
    y=alt.Y("fossil_change_twh:Q", title="Change in fossil fuels (TWh)"),
    color=alt.Color("pattern:N", title="Observed pattern"),
    tooltip=["country:N", alt.Tooltip("renewable_change_twh:Q", format="+,.1f"),
             alt.Tooltip("fossil_change_twh:Q", format="+,.1f")],
)
zero_x = alt.Chart(pd.DataFrame({"zero": [0]})).mark_rule(color="#9ab4bf", strokeDash=[4, 3]).encode(x="zero:Q")
zero_y = alt.Chart(pd.DataFrame({"zero": [0]})).mark_rule(color="#9ab4bf", strokeDash=[4, 3]).encode(y="zero:Q")
chart4 = (scatter + zero_x + zero_y).properties(width=650, height=350,
    title="Changes in renewable and fossil generation, 2010–2024")
display(chart4)

pattern
Both increased                 17
Renewables up, fossils down    13
Name: count, dtype: int64


alt.LayerChart(...)

**Interpretation:** Use the printed pattern counts for the specifically defined top-30 sample. Countries in the upper-right quadrant expanded both renewable and fossil electricity; countries in the lower-right increased renewables while reducing fossil generation. Changes alone cannot establish that one source caused another to rise or fall.

**Follow-up:** Do the 2010 and 2024 endpoints mask variation during the intervening years?

## 7. Case studies: China, United States, Germany and India
**Why these four?** These countries are among the largest producers and illustrate contrasting patterns. Historical trajectories help us see whether two-endpoint differences hide major reversals. Because their systems differ in size, each small multiple has an independently scaled vertical axis; comparisons across countries must therefore be made cautiously.

In [ ]:
case_countries = ["China", "United States", "Germany", "India"]
cases = countries.loc[
    countries["country"].isin(case_countries) & countries["year"].between(2010, 2024)
].copy()
case_long = cases[["country", "year", "renewables_electricity_twh",
                   "fossil_fuels_electricity_twh"]].melt(
    id_vars=["country", "year"], var_name="source", value_name="twh"
).dropna(subset=["twh"])
case_long["source"] = case_long["source"].map({
    "renewables_electricity_twh": "Renewables",
    "fossil_fuels_electricity_twh": "Fossil fuels"
})

chart5 = alt.Chart(case_long).mark_line(strokeWidth=2.5).encode(
    x=alt.X("year:Q", title="Year", axis=alt.Axis(format="d")),
    y=alt.Y("twh:Q", title="Generation (TWh)", scale=alt.Scale(zero=True)),
    color=alt.Color("source:N", title=None, scale=alt.Scale(
        domain=["Renewables", "Fossil fuels"], range=["#18896b", "#cb713f"]
    )),
    tooltip=["country:N", "year:Q", "source:N", alt.Tooltip("twh:Q", format=",.1f")]
).properties(width=300, height=170).facet("country:N", columns=2).resolve_scale(y="independent")
display(chart5)
print("Note: each country has its own y-axis scale; compare directions, not heights.")

alt.FacetChart(...)

Note: each country has its own y-axis scale; compare directions, not heights.


## 8. Conclusions, limitations and final design decisions

**Core story:** Renewable generation has grown rapidly, particularly wind and solar, but a rising renewable share does not automatically imply an absolute decline in fossil generation. National trajectories differ.

**Five complementary panels:** (1) global electricity shares; (2) technology composition; (3) renewable-share changes among 15 major producers; (4) renewable-versus-fossil generation changes for 30 major producers; (5) four contrasting historical trajectories.

**Sampling rule:** Rank countries with complete 2010 and 2024 observations for the required measures by their **2024 electricity generation**. Use the top 15 for the ranking and top 30 for the scatter. This is the identical rule used by `final_visualization.py`.

**Validity and limitations:** Do not sum countries with the World aggregate; electricity ≠ all energy; missing ≠ zero; correlation or simultaneous growth ≠ causation; percentage points ≠ percent; the top 30 is a selected sample rather than all countries. In the faceted comparison, each country's y-axis is independent, so visual slopes are not directly comparable across countries.

**Source:** https://catalog.ourworldindata.org/garden/energy/2026-09-10/owid_energy/owid_energy.csv

**Final artifact:** Run `final_visualization.py` in the same folder as this notebook. On its first run, it downloads the pinned CSV if `owid_energy.csv` is not already available. It generates one PDF and an optional HTML preview. Examine the PDF at 100% zoom to check all labels and the source URL before submission.

### 📊 The 2-Minute Professional Pitch: Explaining my Final Charts

*Read this script out loud during your presentation to explain all 5 panels professionally and concisely in under 2 minutes.*

---

#### 📈 Panel 1: Global Electricity Mix (2000–2025)
* **What I looked at:** The structural market shares of fossil fuels versus renewables over the last quarter-century.
* **My Key Finding:** Fossil share peaked near 68% in the late 2000s and is down to 57%, while renewables rose from 19% to over 33%.
* **Why it's fascinating:** This establishes the macro-transition. But as a data scientist, I immediately saw the catch: a declining *percentage share* doesn't mean absolute fossil generation is falling, because the global power demand pie is expanding.

#### 🗺️ Panel 2: Renewable Generation by Technology
* **What I looked at:** An area chart breaking down absolute global renewable generation in TWh.
* **My Key Finding:** While Hydro remains our historical baseline, the explosive growth over the last decade is driven almost entirely by **Wind** and **Solar**.
* **Why it's fascinating:** We are watching a clear technological inflection point where modern, modular solar and wind are outpacing legacy hydro due to rapidly falling hardware costs.

#### 📊 Panel 3: Change in Renewable Share (Top 15 Countries, 2010–2024)
* **What I looked at:** Net change in renewable market share (in percentage points) for the world's 15 largest power grids.
* **My Key Finding:** The UK, Germany, and Brazil made massive structural shifts (gaining 20 to 35+ percentage points). Meanwhile, grids like Iran or Saudi Arabia remained flat.
* **Why it's fascinating:** It proves that a single 'global average' is a myth. National grids are transitioning at radically different speeds depending on domestic resources and policy.

#### 🎯 Panel 4: Renewable vs. Fossil Generation Changes (Top 30 Countries)
* **What I looked at:** A scatter plot mapping change in renewable TWh against change in fossil TWh. Zero lines divide countries into quadrants.
* **My Key Finding:** Only 13 of the top 30 major producers successfully substituted fossils with renewables. The other 17 countries expanded both to satisfy rising demand.
* **Why it's fascinating:** This is my favorite chart. It completely debunks the assumption that *more renewables automatically equals fewer fossils*. In developing nations, renewables are not yet replacing fossils; they are just absorbing the growth in demand.

#### 📉 Panel 5: Deep-Dive Trajectories (US, China, Germany, India)
* **What I looked at:** Continuous historical paths of the four largest global power generators, using independent Y-axes.
* **My Key Finding:** Germany shows clean substitution; the US shows a peak-and-decline in fossils; China and India are aggressively growing both.
* **Why it's fascinating:** It exposes the danger of looking only at endpoints. China generates more renewable energy than any country on earth, yet they still had to increase fossil generation to fuel their economic expansion.